In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into memory (variables preserved for later steps)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Convert timestamp to timezone-aware datetime in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_log['dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_error['dt'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# --- metric.csv summary ---
metric_count = len(df_metric)
metric_cmdbs = pd.Series(df_metric['cmdb_id'].unique()[:20], name='cmdb_id')  # up to 20 unique cmdb_id
metric_kpis = pd.Series(df_metric['kpi_name'].unique()[:20], name='kpi_name')  # up to 20 unique kpi_name

# Grouped aggregates: count, min, P50, P95, max for each (cmdb_id, kpi_name)
metric_groups = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(count='count',
         min_value='min',
         P50=lambda x: x.quantile(0.5),
         P95=lambda x: x.quantile(0.95),
         max_value='max')
    .reset_index()
    .sort_values('count', ascending=False)
)
# Keep top 20 groups
metric_groups_top20 = metric_groups.head(20)[['cmdb_id','kpi_name','count','min_value','P50','P95','max_value']]

# --- trace.csv summary ---
trace_count = len(df_trace)
trace_cmdbs = pd.Series(df_trace['cmdb_id'].unique()[:20], name='cmdb_id')
trace_names = pd.Series(df_trace['trace_name'].unique()[:20], name='trace_name')

trace_groups = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(count='count',
         min_value='min',
         P50=lambda x: x.quantile(0.5),
         P95=lambda x: x.quantile(0.95),
         max_value='max')
    .reset_index()
    .sort_values('count', ascending=False)
)
trace_groups_top20 = trace_groups.head(20)[['cmdb_id','trace_name','count','min_value','P50','P95','max_value']]

# --- log.csv summary ---
log_count = len(df_log)
log_cmdbs = pd.Series(df_log['cmdb_id'].unique()[:20], name='cmdb_id')
log_names = pd.Series(df_log['log_name'].unique()[:20], name='log_name')

log_groups = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(count='count',
         min_value='min',
         P50=lambda x: x.quantile(0.5),
         P95=lambda x: x.quantile(0.95),
         max_value='max')
    .reset_index()
    .sort_values('count', ascending=False)
)
log_groups_top20 = log_groups.head(20)[['cmdb_id','log_name','count','min_value','P50','P95','max_value']]

# --- error_logs.csv summary ---
error_count = len(df_error)
error_cmdbs = pd.Series(df_error['cmdb_id'].unique()[:20], name='cmdb_id')
error_earliest_dt = df_error['dt'].min() if error_count > 0 else None
error_latest_dt = df_error['dt'].max() if error_count > 0 else None
error_head10 = df_error[['timestamp','cmdb_id','message']].head(10)  # show up to first 10 rows

# Final compact outputs (variables shown in order)
metric_count, metric_cmdbs, metric_kpis, metric_groups_top20, \
trace_count, trace_cmdbs, trace_names, trace_groups_top20, \
log_count, log_cmdbs, log_names, log_groups_top20, \
error_count, error_cmdbs, error_earliest_dt, error_latest_dt, error_head10

```
Out[1]:
```
# Summarize the loaded telemetry inventories in plain English (compact)
summary = (
    "Summary of loaded telemetry files:\n\n"
    "1) metric.csv\n"
    "- Total rows: 144,252\n"
    "- Sample (up to 20) cmdb_id examples: adservice, adservice-0, adservice-1, adservice-2, adservice2, adservice2-0, cartservice, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice, currencyservice-0, currencyservice-1, currencyservice-2\n"
    "- Sample (up to 20) kpi_name examples: app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, app.http.mrt, app.http.rr, app.http.sr, runtime.java_lang_ClassLoading_LoadedClassCount, runtime.java_lang_TotalLoadedClassCount, runtime.java_lang_UnloadedClassCount, runtime.java_lang_ClassLoading_Verbose, runtime.java_lang_Compilation_CompilationTimeMillis, runtime.java_lang_TotalCompilationTime, runtime.java_lang_GarbageCollector_CollectionCount, runtime.java_lang_GarbageCollector_CollectionTime, etc.\n"
    "- Top grouped (cmdb_id, kpi_name) records (top 20 by count): groups are small time-series slices (most top groups have count = 30). Notable top groups include many adservice metrics (e.g., app.grpc.count, app.http.* and many JVM/runtime metrics) and shippingservice2-0 metrics. For these groups the reported sample stats (min, P50, P95, max) were computed from the full file.\n\n"
    "2) trace.csv\n"
    "- Total rows: 36,756\n"
    "- Sample (up to 20) cmdb_id examples: adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice-0, currencyservice-1, currencyservice-2, currencyservice2-0, emailservice-0, emailservice-1, emailservice-2, emailservice2-0\n"
    "- Sample (up to 20) trace_name examples: trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, trace.from_frontend-0.row_count, trace.from_frontend-1.duration_mean, trace.from_frontend-1.duration_p95, etc.\n"
    "- Top grouped (cmdb_id, trace_name) records (top 20 by count): again many top groups have count = 30. Notable top groups include shippingservice2-0 trace.from_frontend2-0.row_count and multiple adservice-* traces (duration and row_count metrics).\n\n"
    "3) log.csv\n"
    "- Total rows: 1,814\n"
    "- Sample (up to 20) cmdb_id examples: adservice-0, adservice-1, adservice-2, cartservice-0, cartservice-1, cartservice-2, checkoutservice-0, checkoutservice-1, checkoutservice-2, currencyservice-0, currencyservice-1, currencyservice-2, emailservice-0, emailservice-1, emailservice-2, frontend-0, frontend-1, frontend-2, paymentservice-0, paymentservice-1\n"
    "- log_name values present (up to 20): log.error_count, log.row_count\n"
    "- Top grouped (cmdb_id, log_name) records (top 20 by count): top groups have count = 30. Example: adservice-0 log.row_count (median ~1427), many services show log.row_count and log.error_count with median error_count = 0 for these groups.\n\n"
    "4) error_logs.csv\n"
    "- Total rows: 1,914\n"
    "- Unique cmdb_id samples (up to 20): frontend-1, frontend-0, frontend-2 (these are the primary sources in this file)\n"
    "- Earliest timestamp (UTC+8): 2022-03-20 11:30:00+08:00\n"
    "- Latest timestamp (UTC+8): 2022-03-20 11:59:58+08:00\n"
    "- First up to 10 rows: all shown rows are frontend components reporting warnings with message text like 'severity: warning, message: failed to retrieve...'.\n\n"
    "Notes/implications:\n"
    "- All aggregated percentiles and group stats were computed over the full respective files (no time filtering).\n"
    "- Many metric/trace/log groups have exactly 30 samples (suggesting ~30-minute slices at 1-minute granularity for those series).\n"
    "- error_logs are concentrated on frontend pods (frontend-0/1/2) in the 11:30–11:59:58+08 window and show repeated 'failed to retrieve' warnings — worth investigating frontend connectivity or upstream dependencies.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(144252, 0              adservice
1            adservice-0
2            adservice-1
3            adservice-2
4             adservice2
5           adservice2-0
6            cartservice
7          cartservice-0
8          cartservice-1
9          cartservice-2
10        cartservice2-0
11       checkoutservice
12     checkoutservice-0
13     checkoutservice-1
14     checkoutservice-2
15    checkoutservice2-0
16       currencyservice
17     currencyservice-0
18     currencyservice-1
19     currencyservice-2
Name: cmdb_id, dtype: object, 0                                        app.grpc.count
1                                          app.grpc.mrt
2                                           app.grpc.rr
3                                           app.grpc.sr
4                                        app.http.count
5                                          app.http.mrt
6                                           app.http.rr
7                                           app.http.sr
8       runtime.java_lang_ClassLoading_LoadedClassCount
9     runtime.java_lang_ClassLoading_TotalLoadedClas...
10    runtime.java_lang_ClassLoading_UnloadedClassCount
11               runtime.java_lang_ClassLoading_Verbose
12    runtime.java_lang_Compilation_CompilationTimeM...
13    runtime.java_lang_Compilation_TotalCompilation...
14    runtime.java_lang_GarbageCollector_CollectionC...
15    runtime.java_lang_GarbageCollector_CollectionC...
16    runtime.java_lang_GarbageCollector_CollectionT...
17    runtime.java_lang_GarbageCollector_CollectionT...
18    runtime.java_lang_GarbageCollector_LastGcInfo_...
19    runtime.java_lang_GarbageCollector_LastGcInfo_...
Name: kpi_name, dtype: object,                  cmdb_id                                           kpi_name  count     min_value           P50           P95     max_value
4817  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30    796.000000   1293.500000   1791.000000   1840.750000
0              adservice                                     app.grpc.count     30    205.000000    231.000000    249.100000    262.000000
1              adservice                                       app.grpc.mrt     30      0.002218      0.002295      0.002368      0.002391
2              adservice                                        app.grpc.rr     30    100.000000    100.000000    100.000000    100.000000
3              adservice                                        app.grpc.sr     30     52.000000     55.565298     58.016193     58.715596
4              adservice                                     app.http.count     30     24.000000     24.000000     24.000000     24.000000
5              adservice                                       app.http.mrt     30      0.008425      0.008854      0.009102      0.009144
6              adservice                                        app.http.rr     30    100.000000    100.000000    100.000000    100.000000
7              adservice                                        app.http.sr     30    100.000000    100.000000    100.000000    100.000000
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount     30   4990.000000   4990.000000   4990.000000   4990.000000
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     30   4990.000000   4990.000000   4990.000000   4990.000000
10             adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     30      0.000000      0.000000      0.000000      0.000000
11             adservice             runtime.java_lang_ClassLoading_Verbose     30      0.000000      0.000000      0.000000      0.000000
12             adservice  runtime.java_lang_Compilation_CompilationTimeM...     30      1.000000      1.000000      1.000000      1.000000
13             adservice  runtime.java_lang_Compilation_TotalCompilation...     30  43234.000000  43319.000000  43334.000000  43334.000000
14             adservice  runtime.java_lang_GarbageCollector_CollectionC...     30  33138.666667  33268.750000  33386.200000  33399.166667
15             adservice  runtime.java_lang_GarbageCollector_CollectionC...     30      3.000000      3.000000      3.000000      3.000000
16             adservice  runtime.java_lang_GarbageCollector_CollectionT...     30  52112.333333  52315.416667  52496.670833  52517.583333
4778  shippingservice2-0  container.node-5.container_network_receive_MB....     30      0.023887      0.032258      0.178832      0.297482
4779  shippingservice2-0  container.node-5.container_network_receive_err...     30      0.000000      0.000000      0.000000      0.000000, 36756, 0            adservice-0
1            adservice-1
2            adservice-2
3           adservice2-0
4          cartservice-0
5          cartservice-1
6          cartservice-2
7         cartservice2-0
8      checkoutservice-0
9      checkoutservice-1
10     checkoutservice-2
11    checkoutservice2-0
12     currencyservice-0
13     currencyservice-1
14     currencyservice-2
15    currencyservice2-0
16        emailservice-0
17        emailservice-1
18        emailservice-2
19       emailservice2-0
Name: cmdb_id, dtype: object, 0            trace.from_frontend-0.duration_mean
1             trace.from_frontend-0.duration_p95
2               trace.from_frontend-0.error_rate
3                trace.from_frontend-0.row_count
4            trace.from_frontend-1.duration_mean
5             trace.from_frontend-1.duration_p95
6               trace.from_frontend-1.error_rate
7                trace.from_frontend-1.row_count
8            trace.from_frontend-2.duration_mean
9             trace.from_frontend-2.duration_p95
10              trace.from_frontend-2.error_rate
11               trace.from_frontend-2.row_count
12          trace.from_frontend2-0.duration_mean
13           trace.from_frontend2-0.duration_p95
14             trace.from_frontend2-0.error_rate
15              trace.from_frontend2-0.row_count
16    trace.from_checkoutservice-0.duration_mean
17     trace.from_checkoutservice-0.duration_p95
18       trace.from_checkoutservice-0.error_rate
19        trace.from_checkoutservice-0.row_count
Name: trace_name, dtype: object,                  cmdb_id                           trace_name  count  min_value        P50        P95  max_value
1235  shippingservice2-0     trace.from_frontend2-0.row_count     30   4.000000  13.500000  21.000000  22.000000
0            adservice-0  trace.from_frontend-0.duration_mean     30   0.000011   0.000012   0.000017   0.000020
1            adservice-0   trace.from_frontend-0.duration_p95     30   0.000012   0.000014   0.000020   0.000022
2            adservice-0     trace.from_frontend-0.error_rate     30   0.000000   0.000000   0.000000   0.000000
3            adservice-0      trace.from_frontend-0.row_count     30  19.000000  23.000000  27.100000  30.000000
4            adservice-0  trace.from_frontend-1.duration_mean     30   0.000011   0.000012   0.000022   0.000023
5            adservice-0   trace.from_frontend-1.duration_p95     30   0.000012   0.000014   0.000033   0.000049
6            adservice-0     trace.from_frontend-1.error_rate     30   0.000000   0.000000   0.000000   0.000000
7            adservice-0      trace.from_frontend-1.row_count     30  13.000000  18.000000  21.000000  22.000000
8            adservice-0  trace.from_frontend-2.duration_mean     30   0.000012   0.000012   0.000014   0.000015
9            adservice-0   trace.from_frontend-2.duration_p95     30   0.000013   0.000014   0.000020   0.000023
10           adservice-0     trace.from_frontend-2.error_rate     30   0.000000   0.000000   0.000000   0.000000
11           adservice-0      trace.from_frontend-2.row_count     30  16.000000  23.000000  26.550000  30.000000
12           adservice-1  trace.from_frontend-0.duration_mean     30   0.000012   0.000013   0.000015   0.000022
13           adservice-1   trace.from_frontend-0.duration_p95     30   0.000013   0.000017   0.000022   0.000024
14           adservice-1     trace.from_frontend-0.error_rate     30   0.000000   0.000000   0.000000   0.000000
15           adservice-1      trace.from_frontend-0.row_count     30  19.000000  23.000000  27.100000  30.000000
16           adservice-1  trace.from_frontend-1.duration_mean     30   0.000012   0.000013   0.000019   0.000022
17           adservice-1   trace.from_frontend-1.duration_p95     30   0.000012   0.000015   0.000024   0.000039
18           adservice-1     trace.from_frontend-1.error_rate     30   0.000000   0.000000   0.000000   0.000000, 1814, 0           adservice-0
1           adservice-1
2           adservice-2
3         cartservice-0
4         cartservice-1
5         cartservice-2
6     checkoutservice-0
7     checkoutservice-1
8     checkoutservice-2
9     currencyservice-0
10    currencyservice-1
11    currencyservice-2
12       emailservice-0
13       emailservice-1
14       emailservice-2
15           frontend-0
16           frontend-1
17           frontend-2
18     paymentservice-0
19     paymentservice-1
Name: cmdb_id, dtype: object, 0    log.error_count
1      log.row_count
Name: log_name, dtype: object,               cmdb_id         log_name  count  min_value     P50      P95  max_value
0         adservice-0  log.error_count     30        0.0     0.0     0.00        0.0
1         adservice-0    log.row_count     30     1246.0  1427.0  1504.20     1518.0
2         adservice-1  log.error_count     30        0.0     0.0     0.00        0.0
3         adservice-1    log.row_count     30      114.0   130.0   137.10      139.0
4         adservice-2  log.error_count     30        0.0     0.0     0.00        0.0
5         adservice-2    log.row_count     30      112.0   128.0   136.00      142.0
6       cartservice-0  log.error_count     30        0.0     0.0     0.00        0.0
7       cartservice-0    log.row_count     30      966.0  1021.0  1074.55     1077.0
8       cartservice-1  log.error_count     30        0.0     0.0     0.00        0.0
9       cartservice-1    log.row_count     30      930.0  1024.5  1073.50     1078.0
10      cartservice-2  log.error_count     30        0.0     0.0     0.00        0.0
11      cartservice-2    log.row_count     30      931.0  1023.0  1076.60     1085.0
12  checkoutservice-0  log.error_count     30        0.0     0.0     0.00        0.0
13  checkoutservice-0    log.row_count     30       37.0    70.0    94.55       99.0
14  checkoutservice-1  log.error_count     30        0.0     0.0     0.00        0.0
15  checkoutservice-1    log.row_count     30       34.0    76.0    85.00       99.0
16  checkoutservice-2  log.error_count     30        0.0     0.0     0.00        0.0
17  checkoutservice-2    log.row_count     30       37.0    68.0    97.10       99.0
18  currencyservice-0  log.error_count     30        0.0     0.0     0.00        0.0
19  currencyservice-0    log.row_count     30      814.0   887.0   953.20      959.0, 1914, 0    frontend-1
1    frontend-0
2    frontend-2
Name: cmdb_id, dtype: object, Timestamp('2022-03-20 11:30:00+0800', tz='Asia/Shanghai'), Timestamp('2022-03-20 11:59:58+0800', tz='Asia/Shanghai'),     timestamp     cmdb_id                                            message
0  1647747000  frontend-1  severity: warning, message: failed to retrieve...
1  1647747000  frontend-1  severity: warning, message: failed to retrieve...
2  1647747002  frontend-0  severity: warning, message: failed to retrieve...
3  1647747003  frontend-0  severity: warning, message: failed to retrieve...
4  1647747005  frontend-2  severity: warning, message: failed to retrieve...
5  1647747005  frontend-1  severity: warning, message: failed to retrieve...
6  1647747006  frontend-2  severity: warning, message: failed to retrieve...
7  1647747007  frontend-0  severity: warning, message: failed to retrieve...
8  1647747008  frontend-1  severity: warning, message: failed to retrieve...
9  1647747011  frontend-0  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

# Reuse existing DataFrames: df_metric, df_trace, df_log, tz, and dt columns created earlier.
import pandas as pd

# Ensure tz variable exists
try:
    tz
except NameError:
    import pytz
    tz = pytz.timezone('Asia/Shanghai')

# Incident window in UTC+8
window_start = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
window_end = pd.Timestamp('2022-03-20 11:59:59', tz=tz)

# Helper: lower-is-bad substrings (case-insensitive)
lower_is_bad_substrs = ['sr', 'success', 'ss', 'rate', 'rr']

def is_lower_is_bad(name):
    n = str(name).lower()
    return any(sub in n for sub in lower_is_bad_substrs)

# -------------------------
# 1) Compute global thresholds (P95 and P05) for full series (no time filtering)
# -------------------------

# Metric thresholds (group by cmdb_id, kpi_name)
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(group_count_total='count',
         threshold_P95=lambda x: x.quantile(0.95),
         threshold_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Trace thresholds
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(group_count_total='count',
         threshold_P95=lambda x: x.quantile(0.95),
         threshold_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Log thresholds
log_thresholds = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(group_count_total='count',
         threshold_P95=lambda x: x.quantile(0.95),
         threshold_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# -------------------------
# 2) Detect anomalies inside the incident window
# -------------------------

# Metric window points joined with thresholds
metric_window = df_metric[(df_metric['dt'] >= window_start) & (df_metric['dt'] <= window_end)].copy()
metric_window = metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Determine direction per group (based on kpi_name)
metric_window['direction'] = metric_window['kpi_name'].apply(lambda n: 'low' if is_lower_is_bad(n) else 'high')

# Anomaly rule: high -> value >= P95 ; low -> value <= P05
metric_window['anomalous'] = metric_window.apply(
    lambda r: (r['value'] >= r['threshold_P95']) if r['direction']=='high' else (r['value'] <= r['threshold_P05']),
    axis=1
)

# Keep full intermediate df
metric_anoms_df = metric_window.copy()

# Aggregate anomalies per group
metric_anom_groups = (
    metric_window[metric_window['anomalous']]
    .groupby(['cmdb_id','kpi_name','direction'])
    .agg(anomaly_count_within_window=('anomalous','sum'),
         earliest_anomalous_timestamp_within_window=('dt','min'),
         max_value_within_window=('value','max'),
         min_value_within_window=('value','min'))
    .reset_index()
    .merge(metric_thresholds, left_on=['cmdb_id','kpi_name'], right_on=['cmdb_id','kpi_name'], how='left')
    .rename(columns={'kpi_name':'name'})
)

# -------------------------
# Trace anomalies
# -------------------------
trace_window = df_trace[(df_trace['dt'] >= window_start) & (df_trace['dt'] <= window_end)].copy()
trace_window = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
trace_window['direction'] = trace_window['trace_name'].apply(lambda n: 'low' if is_lower_is_bad(n) else 'high')
trace_window['anomalous'] = trace_window.apply(
    lambda r: (r['value'] >= r['threshold_P95']) if r['direction']=='high' else (r['value'] <= r['threshold_P05']),
    axis=1
)
trace_anoms_df = trace_window.copy()
trace_anom_groups = (
    trace_window[trace_window['anomalous']]
    .groupby(['cmdb_id','trace_name','direction'])
    .agg(anomaly_count_within_window=('anomalous','sum'),
         earliest_anomalous_timestamp_within_window=('dt','min'),
         max_value_within_window=('value','max'),
         min_value_within_window=('value','min'))
    .reset_index()
    .merge(trace_thresholds, left_on=['cmdb_id','trace_name'], right_on=['cmdb_id','trace_name'], how='left')
    .rename(columns={'trace_name':'name'})
)

# -------------------------
# Log anomalies
# -------------------------
log_window = df_log[(df_log['dt'] >= window_start) & (df_log['dt'] <= window_end)].copy()
log_window = log_window.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')

# For log.error_count treat spikes as higher-is-bad; otherwise apply substring rule
def log_direction(name):
    n = str(name).lower()
    if 'error_count' in n:
        return 'high'
    return 'low' if is_lower_is_bad(n) else 'high'

log_window['direction'] = log_window['log_name'].apply(log_direction)
log_window['anomalous'] = log_window.apply(
    lambda r: (r['value'] >= r['threshold_P95']) if r['direction']=='high' else (r['value'] <= r['threshold_P05']),
    axis=1
)
log_anoms_df = log_window.copy()
log_anom_groups = (
    log_window[log_window['anomalous']]
    .groupby(['cmdb_id','log_name','direction'])
    .agg(anomaly_count_within_window=('anomalous','sum'),
         earliest_anomalous_timestamp_within_window=('dt','min'),
         max_value_within_window=('value','max'),
         min_value_within_window=('value','min'))
    .reset_index()
    .merge(log_thresholds, left_on=['cmdb_id','log_name'], right_on=['cmdb_id','log_name'], how='left')
    .rename(columns={'log_name':'name'})
)

# -------------------------
# 3) Combine and produce compact ranked table (limit 20 rows)
# -------------------------
# Standardize columns: source, cmdb_id, name, group_count_total, threshold_P95, threshold_P05, anomaly_count_within_window, earliest..., max..., min..., direction
def prepare_summary(df, source_label):
    df2 = df.copy()
    df2['source'] = source_label
    # choose and reorder columns
    return df2[[
        'source',
        'cmdb_id',
        'name',
        'group_count_total',
        'threshold_P95',
        'threshold_P05',
        'anomaly_count_within_window',
        'earliest_anomalous_timestamp_within_window',
        'max_value_within_window',
        'min_value_within_window',
        'direction'
    ]]

metric_summary = prepare_summary(metric_anom_groups, 'metric') if not metric_anom_groups.empty else pd.DataFrame(columns=[
    'source','cmdb_id','name','group_count_total','threshold_P95','threshold_P05','anomaly_count_within_window',
    'earliest_anomalous_timestamp_within_window','max_value_within_window','min_value_within_window','direction'
])
trace_summary = prepare_summary(trace_anom_groups, 'trace') if not trace_anom_groups.empty else metric_summary.iloc[0:0].copy()
log_summary = prepare_summary(log_anom_groups, 'log') if not log_anom_groups.empty else metric_summary.iloc[0:0].copy()

combined = pd.concat([metric_summary, trace_summary, log_summary], ignore_index=True)

# Only keep groups with anomalies (they should already be), sort by anomaly_count desc, keep top 20
if not combined.empty:
    combined_sorted = combined.sort_values('anomaly_count_within_window', ascending=False)
    anomalies_summary_top20 = combined_sorted.head(20).reset_index(drop=True)
else:
    anomalies_summary_top20 = combined.copy()

# Display only the compact table (max 20 rows)
anomalies_summary_top20

```
Out[2]:
```
# Compose a plain-English compact summary based on the detected anomalies (use existing `anomalies_summary_top20`)
log_groups = anomalies_summary_top20[anomalies_summary_top20['source']=='log']
trace_groups = anomalies_summary_top20[anomalies_summary_top20['source']=='trace']
metric_groups = anomalies_summary_top20[anomalies_summary_top20['source']=='metric']

summary = (
    f"Anomaly summary for incident window 2022-03-20 11:30:00 to 2022-03-20 11:59:59 (UTC+8):\n\n"
    f"- Top groups reported (displaying up to 20 groups): {anomalies_summary_top20.shape[0]} groups.\n"
    f"- Most anomalies are from log.error_count groups across many services (log entries: {len(log_groups)} of top 20). "
    f"Examples include shippingservice-2, currencyservice-1/2, emailservice-0/1, checkoutservice-0/1/2, cartservice-0/1/2, adservice-0/1/2. "
    f"Each of these groups shows group_count_total=30, threshold_P95=0.0, and anomaly_count_within_window=30 (every point flagged). \n\n"
    f"- One trace group (shippingservice2-0 trace.from_frontend2-0.error_rate) is flagged as 'low' with P05=0.0 and anomaly_count=30 (all points at 0).\n\n"
    f"- A few metric groups related to JVM garbage-collector LastGcInfo for adservice are flagged (values equal their P95), each with 30 samples and anomaly_count=30.\n\n"
    f"Interpretation and note: many groups are flagged because their global thresholds (P95 or P05) equal the observed constant value (e.g., 0.0 or identical GC values), so all points inside the window meet the anomaly rule. "
    f"This suggests the anomalies are driven by threshold edge-cases (zero or constant-series) rather than clear spikes/drops; recommend reviewing the full-series distributions for these groups and verifying the thresholding logic before acting.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

source             cmdb_id                                               name  group_count_total  threshold_P95  threshold_P05  anomaly_count_within_window earliest_anomalous_timestamp_within_window  max_value_within_window  min_value_within_window direction
0      log   shippingservice-2                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
1      log   currencyservice-1                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
2      log   currencyservice-2                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
3      log      emailservice-0                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
4      log      emailservice-1                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
5      log   checkoutservice-0                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
6      log   checkoutservice-1                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
7      log   checkoutservice-2                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
8      log   currencyservice-0                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
9      log         adservice-2                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
10     log       cartservice-0                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
11     log       cartservice-1                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
12     log       cartservice-2                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
13   trace  shippingservice2-0                  trace.from_frontend2-0.error_rate                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0       low
14     log         adservice-0                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
15     log         adservice-1                                    log.error_count                 30            0.0            0.0                           30                  2022-03-20 11:30:00+08:00                      0.0                      0.0      high
16  metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 30           -1.0           -1.0                           30                  2022-03-20 11:30:00+08:00                     -1.0                     -1.0      high
17  metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 30      5242880.0      5242880.0                           30                  2022-03-20 11:30:00+08:00                5242880.0                5242880.0      high
18  metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 30      5242880.0      5242880.0                           30                  2022-03-20 11:30:00+08:00                5242880.0                5242880.0      high
19  metric           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 30    104857600.0    104857600.0                           30                  2022-03-20 11:30:00+08:00              104857600.0              104857600.0      high```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Ensure timezone and dt column exist
tz = pytz.timezone('Asia/Shanghai')
if 'df_error' not in globals():
    df_error = pd.read_csv("error_logs.csv")
if 'dt' not in df_error.columns:
    df_error['dt'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure timezone-aware and in UTC+8
    df_error['dt'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive) in UTC+8
window_start = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
window_end   = pd.Timestamp('2022-03-20 11:59:59', tz=tz)

# Filter error logs to the incident window and store for follow-ups
error_window_df = df_error[(df_error['dt'] >= window_start) & (df_error['dt'] <= window_end)].copy()

# 1) Per-cmdb_id summary in window: count, earliest, latest (limit top 20 by count)
group_summary = (
    error_window_df
    .groupby('cmdb_id')['dt']
    .agg(count='count',
         earliest_timestamp=lambda x: x.min(),
         latest_timestamp=lambda x: x.max())
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# 2) Global earliest and latest timestamps among all rows in the filtered window
global_earliest = error_window_df['dt'].min() if not error_window_df.empty else None
global_latest = error_window_df['dt'].max() if not error_window_df.empty else None

# 3) For frontend-0, frontend-1, frontend-2: up to first 10 rows each (ordered by timestamp ascending) showing timestamp and message
frontends = ['frontend-0','frontend-1','frontend-2']
frontend_samples = (
    error_window_df[error_window_df['cmdb_id'].isin(frontends)]
    .sort_values(['cmdb_id','dt'])
    .groupby('cmdb_id')
    .head(10)
    .loc[:, ['cmdb_id','timestamp','dt','message']]
    .sort_values(['cmdb_id','dt'])
    .reset_index(drop=True)
)

# Return the three compact outputs (and keep error_window_df in kernel for follow-ups)
group_summary, global_earliest, global_latest, frontend_samples

```
Out[3]:
```
Filtered error logs in the incident window (2022-03-20 11:30:00 → 11:59:59 UTC+8):

1) Per-cmdb_id counts and time range (sorted by count):
- frontend-0: count = 692, earliest = 2022-03-20 11:30:02+08:00, latest = 2022-03-20 11:59:55+08:00  
- frontend-2: count = 688, earliest = 2022-03-20 11:30:05+08:00, latest = 2022-03-20 11:59:58+08:00  
- frontend-1: count = 534, earliest = 2022-03-20 11:30:00+08:00, latest = 2022-03-20 11:59:56+08:00

2) Global earliest and latest in the filtered window:
- Earliest = 2022-03-20 11:30:00+08:00  
- Latest   = 2022-03-20 11:59:58+08:00

3) Sample frontend messages (first rows for frontend-0/1/2):
- The logged messages for frontend-0, frontend-1 and frontend-2 are largely the same repeated warning:  
  "severity: warning, message: failed to retrieve..."  
- These repeated warnings begin right at the start of the window and occur frequently (hundreds of entries per frontend).

Interpretation / next steps (concise):
- Error logs are concentrated on frontend pods (0/1/2) throughout the entire 30-minute window, with frequent repeated "failed to retrieve" warnings.  
- Investigate frontend upstream dependencies (e.g., backend services, network connectivity, DNS) and correlate with traces/metrics for those frontends to find root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  count        earliest_timestamp          latest_timestamp
0  frontend-0    692 2022-03-20 11:30:02+08:00 2022-03-20 11:59:55+08:00
2  frontend-2    688 2022-03-20 11:30:05+08:00 2022-03-20 11:59:58+08:00
1  frontend-1    534 2022-03-20 11:30:00+08:00 2022-03-20 11:59:56+08:00, Timestamp('2022-03-20 11:30:00+0800', tz='Asia/Shanghai'), Timestamp('2022-03-20 11:59:58+0800', tz='Asia/Shanghai'),        cmdb_id   timestamp                        dt                                            message
0   frontend-0  1647747002 2022-03-20 11:30:02+08:00  severity: warning, message: failed to retrieve...
1   frontend-0  1647747003 2022-03-20 11:30:03+08:00  severity: warning, message: failed to retrieve...
2   frontend-0  1647747007 2022-03-20 11:30:07+08:00  severity: warning, message: failed to retrieve...
3   frontend-0  1647747011 2022-03-20 11:30:11+08:00  severity: warning, message: failed to retrieve...
4   frontend-0  1647747014 2022-03-20 11:30:14+08:00  severity: warning, message: failed to retrieve...
..         ...         ...                       ...                                                ...
25  frontend-2  1647747023 2022-03-20 11:30:23+08:00  severity: warning, message: failed to retrieve...
26  frontend-2  1647747026 2022-03-20 11:30:26+08:00  severity: warning, message: failed to retrieve...
27  frontend-2  1647747029 2022-03-20 11:30:29+08:00  severity: warning, message: failed to retrieve...
28  frontend-2  1647747031 2022-03-20 11:30:31+08:00  severity: warning, message: failed to retrieve...
29  frontend-2  1647747036 2022-03-20 11:30:36+08:00  severity: warning, message: failed to retrieve...

[30 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# ensure timezone variable
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure df_trace exists and has timezone-aware dt
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (UTC+8)
window_start = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
window_end   = pd.Timestamp('2022-03-20 11:59:59', tz=tz)

# Filter by trace_name containing 'from_frontend' or 'from_frontend2' (case-sensitive) and by window
mask_name = df_trace['trace_name'].str.contains('from_frontend|from_frontend2', regex=True)
trace_frontend_window_df = df_trace[mask_name & (df_trace['dt'] >= window_start) & (df_trace['dt'] <= window_end)].copy()

# Group and aggregate
trace_frontend_summary = (
    trace_frontend_window_df
    .groupby(['cmdb_id','trace_name'])
    .agg(
        count_in_window=('value','count'),
        min_value=('value','min'),
        median_value=('value', lambda x: x.quantile(0.5)),
        max_value=('value','max'),
        earliest_timestamp_in_window=('dt','min')
    )
    .reset_index()
    .sort_values('count_in_window', ascending=False)
)

# Compact top 20
trace_frontend_summary_top20 = trace_frontend_summary.head(20)

# Return the compact table
trace_frontend_summary_top20

```
Out[4]:
```
Summary of trace data for 2022-03-20 11:30:00 → 11:59:59 (UTC+8), filtered to trace_name containing "from_frontend" or "from_frontend2":

- Results returned (top groups, max 20 rows): most (all shown) groups have 30 samples inside the 30‑minute window (count_in_window = 30), indicating per-minute series covering the full window.

- Notable group (highest count shown):
  - shippingservice2-0 / trace.from_frontend2-0.row_count — count = 30, min = 4.0, median = 13.5, max = 22.0, earliest = 2022-03-20 11:30:00+08:00.

- Many adservice* groups are present with full 30 samples, examples:
  - adservice-0 / trace.from_frontend-0.row_count — min 19, median 23, max 30, earliest 11:30:00+08:00.
  - adservice-0 / duration_* and p95 traces — durations are very small (median ~0.000012s).
  - adservice-0/1/2 error_rate traces are all 0.0 (min/median/max = 0.0) in this window.

- Overall pattern:
  - Row counts from frontends vary by target (e.g., shippingservice2-0 median 13.5, adservice row_count medians ~18–23).
  - Latencies (duration_mean/p95) for adservice traces are extremely low in this window.
  - Error rates reported by these traces are effectively zero for the listed groups during the window.

- Follow-up: the filtered trace rows are stored in kernel variable trace_frontend_window_df for further analysis or correlation with frontend error logs.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                           trace_name  count_in_window  min_value  median_value  max_value earliest_timestamp_in_window
279  shippingservice2-0     trace.from_frontend2-0.row_count               30   4.000000     13.500000  22.000000    2022-03-20 11:30:00+08:00
0           adservice-0  trace.from_frontend-0.duration_mean               30   0.000011      0.000012   0.000020    2022-03-20 11:30:00+08:00
1           adservice-0   trace.from_frontend-0.duration_p95               30   0.000012      0.000014   0.000022    2022-03-20 11:30:00+08:00
2           adservice-0     trace.from_frontend-0.error_rate               30   0.000000      0.000000   0.000000    2022-03-20 11:30:00+08:00
3           adservice-0      trace.from_frontend-0.row_count               30  19.000000     23.000000  30.000000    2022-03-20 11:30:00+08:00
4           adservice-0  trace.from_frontend-1.duration_mean               30   0.000011      0.000012   0.000023    2022-03-20 11:30:00+08:00
5           adservice-0   trace.from_frontend-1.duration_p95               30   0.000012      0.000014   0.000049    2022-03-20 11:30:00+08:00
6           adservice-0     trace.from_frontend-1.error_rate               30   0.000000      0.000000   0.000000    2022-03-20 11:30:00+08:00
7           adservice-0      trace.from_frontend-1.row_count               30  13.000000     18.000000  22.000000    2022-03-20 11:30:00+08:00
8           adservice-0  trace.from_frontend-2.duration_mean               30   0.000012      0.000012   0.000015    2022-03-20 11:30:00+08:00
9           adservice-0   trace.from_frontend-2.duration_p95               30   0.000013      0.000014   0.000023    2022-03-20 11:30:00+08:00
10          adservice-0     trace.from_frontend-2.error_rate               30   0.000000      0.000000   0.000000    2022-03-20 11:30:00+08:00
11          adservice-0      trace.from_frontend-2.row_count               30  16.000000     23.000000  30.000000    2022-03-20 11:30:00+08:00
12          adservice-1  trace.from_frontend-0.duration_mean               30   0.000012      0.000013   0.000022    2022-03-20 11:30:00+08:00
13          adservice-1   trace.from_frontend-0.duration_p95               30   0.000013      0.000017   0.000024    2022-03-20 11:30:00+08:00
14          adservice-1     trace.from_frontend-0.error_rate               30   0.000000      0.000000   0.000000    2022-03-20 11:30:00+08:00
15          adservice-1      trace.from_frontend-0.row_count               30  19.000000     23.000000  30.000000    2022-03-20 11:30:00+08:00
16          adservice-1  trace.from_frontend-1.duration_mean               30   0.000012      0.000013   0.000022    2022-03-20 11:30:00+08:00
17          adservice-1   trace.from_frontend-1.duration_p95               30   0.000012      0.000015   0.000039    2022-03-20 11:30:00+08:00
18          adservice-1     trace.from_frontend-1.error_rate               30   0.000000      0.000000   0.000000    2022-03-20 11:30:00+08:00```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse df_metric if available, otherwise load
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")

# Target frontends
targets = ["frontend-0","frontend-1","frontend-2"]

# Compute full-series counts per (cmdb_id, kpi_name)
_k = df_metric.groupby(['cmdb_id','kpi_name'])['value'].count().reset_index(name='full_series_count')

# Filter to the target cmdb_ids
frontend_kpis = _k[_k['cmdb_id'].isin(targets)].copy()

# Limit to first 100 unique kpi_name values (sorted alphabetically for determinism)
first100_kpis = sorted(frontend_kpis['kpi_name'].unique())[:100]

# Keep only rows whose kpi_name is in the first100 list, sort by cmdb_id then kpi_name
frontend_kpi_counts = (
    frontend_kpis[frontend_kpis['kpi_name'].isin(first100_kpis)]
    .sort_values(['cmdb_id','kpi_name'])
    .reset_index(drop=True)
)

# Store the resulting table in kernel variable `frontend_kpi_counts` and display up to 100 rows
frontend_kpi_counts.head(100)

```
Out[5]:
```
Summary of KPI inventory for frontend-0/1/2:

- I extracted up to 100 unique kpi_name entries for cmdb_id in ["frontend-0","frontend-1","frontend-2"] and stored the result in the kernel variable frontend_kpi_counts.
- The returned compact table (first 100 rows) lists columns: cmdb_id, kpi_name, full_series_count, sorted by cmdb_id then kpi_name.
- Observation: for the displayed rows each frontend KPI has full_series_count = 30 (i.e., 30 samples per KPI in the metric file).
- Example KPI names include: container.node-5.container_cpu_cfs_periods, container.node-5.container_cpu_load_average_10s, container.node-5.container_cpu_system_seconds, and many mesh.source.frontend.* Istio metrics.
- Next step: frontend_kpi_counts is ready for downstream analysis (thresholds, anomalies, or correlation with traces/logs).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  full_series_count
0   frontend-0         container.node-5.container_cpu_cfs_periods                 30
1   frontend-0  container.node-5.container_cpu_cfs_throttled_p...                 30
2   frontend-0  container.node-5.container_cpu_cfs_throttled_s...                 30
3   frontend-0    container.node-5.container_cpu_load_average_10s                 30
4   frontend-0      container.node-5.container_cpu_system_seconds                 30
..         ...                                                ...                ...
95  frontend-0  mesh.source.frontend.adservice.istio_response_...                 30
96  frontend-0  mesh.source.frontend.adservice.istio_response_...                 30
97  frontend-0  mesh.source.frontend.adservice.istio_response_...                 30
98  frontend-0  mesh.source.frontend.cartservice.istio_request...                 30
99  frontend-0  mesh.source.frontend.cartservice.istio_request...                 30

[100 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing df_metric and tz if present, otherwise load/define
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware datetime column 'dt' in UTC+8
df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Targets and incident window (UTC+8)
targets = ["frontend-0","frontend-1","frontend-2"]
window_start = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
window_end   = pd.Timestamp('2022-03-20 11:59:59', tz=tz)

# Helper for lower-is-bad detection
lower_is_bad_substrs = ['sr','success','ss','rate','rr']
def is_lower_is_bad(name):
    n = str(name).lower()
    return any(sub in n for sub in lower_is_bad_substrs)

# 1) Compute global thresholds (P95 and P05) using full series (no time filtering)
frontend_metric_thresholds = (
    df_metric[df_metric['cmdb_id'].isin(targets)]
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(full_series_count='count',
         threshold_P95=lambda x: x.quantile(0.95),
         threshold_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric points to the incident window and join thresholds
frontend_metric_window = df_metric[
    (df_metric['cmdb_id'].isin(targets)) &
    (df_metric['dt'] >= window_start) &
    (df_metric['dt'] <= window_end)
].copy()

# Merge thresholds into window points
frontend_metric_window = frontend_metric_window.merge(
    frontend_metric_thresholds,
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Determine direction per kpi_name
frontend_metric_window['direction'] = frontend_metric_window['kpi_name'].apply(
    lambda n: 'low' if is_lower_is_bad(n) else 'high'
)

# Flag anomalies per rules: high -> value >= P95 ; low -> value <= P05
frontend_metric_window['anomalous'] = frontend_metric_window.apply(
    lambda r: (r['value'] >= r['threshold_P95']) if r['direction']=='high' else (r['value'] <= r['threshold_P05']),
    axis=1
)

# Store detailed flagged anomalies DataFrame for follow-ups
frontend_metric_anoms_df = frontend_metric_window.copy()

# 3) Aggregate anomalies per (cmdb_id, kpi_name)
frontend_anom_summary = (
    frontend_metric_anoms_df[frontend_metric_anoms_df['anomalous']]
    .groupby(['cmdb_id','kpi_name','direction'])
    .agg(
        anomaly_count_in_window=('anomalous','sum'),
        earliest_anomalous_timestamp_in_window=('dt','min'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min')
    ).reset_index()
    .merge(frontend_metric_thresholds, left_on=['cmdb_id','kpi_name'], right_on=['cmdb_id','kpi_name'], how='left')
    .loc[:, [
        'cmdb_id',
        'kpi_name',
        'full_series_count',
        'threshold_P95',
        'threshold_P05',
        'anomaly_count_in_window',
        'earliest_anomalous_timestamp_in_window',
        'max_value_in_window',
        'min_value_in_window',
        'direction'
    ]]
)

# Sort by anomaly_count_in_window descending and keep top 20 (compact)
frontend_anom_summary_top20 = frontend_anom_summary.sort_values(
    'anomaly_count_in_window', ascending=False
).head(20).reset_index(drop=True)

# Return only the compact table (and keep frontend_metric_anoms_df in kernel)
frontend_anom_summary_top20

```
Out[6]:
```
Summary of frontend metric anomaly detection for the incident window (2022-03-20 11:30:00 → 11:59:59 UTC+8):

- What was done: For frontend-0/1/2 we computed global thresholds (P95 and P05) using the entire metric series, then flagged points inside the 30‑minute incident window as anomalous if they met the rules (>=P95 for "high" KPIs; <=P05 for KPIs treated as "low").

- Key result (compact):
  - Many frontend (cmdb_id, kpi_name) groups had anomalies and are listed (top 20 shown). Nearly all flagged groups have:
    - full_series_count = 30 (i.e., 30 samples in the full file),
    - anomaly_count_in_window = 30 (every point in the window flagged),
    - earliest anomalous timestamp = 2022-03-20 11:30:00+08:00 (or within seconds of window start).
  - Examples from the top results:
    - frontend-2 · mesh.source.frontend.productcatalogservice... — full_series_count=30, P95=0.0, P05=0.0, anomalies=30, direction=high.
    - frontend-2 · mesh.source.frontend.cartservice.istio_request... — full_series_count=30, P95=0.0, anomalies=30, direction=high.
    - frontend-0 · container.node-5.container_spec_cpu_period — full_series_count=30, threshold_P95=100000.0, anomalies=30, direction=high.
    - Several container task-state / thread / mesh metrics for frontend-0 and frontend-2 are also flagged, typically with threshold values 0.0 and all window points equal 0.0.
  - Direction: most flagged groups are labeled "high" under the applied rule; a few metrics labeled "low" appear where the substring rule matched.

- Interpretation / caveat:
  - Many anomalies stem from threshold edge-cases: thresholds (P95 or P05) equal the observed constant values (often 0.0 or a fixed integer), so every point in the window meets the anomaly condition. This typically indicates constant or zero-valued series rather than a true spike/drop.
  - Because the global thresholding used the full series, groups with no variability will produce P95==P05==value and thus all points get flagged.

- Suggested next steps:
  - Inspect full-series distributions for the flagged KPIs (to distinguish true deviations from constant/zero series).
  - Cross-correlate these metric anomalies with frontend error logs and frontend→service traces (we already observed repeated "failed to retrieve" warnings in frontend logs and trace activity for frontends) to identify likely root causes.
  - Consider adjusting thresholding logic for constant/zero series (e.g., require variance or a minimum interquartile range before applying simple percentile rules).

- Note: the detailed flagged-anomalies DataFrame is stored in the kernel variable frontend_metric_anoms_df for follow-up investigation.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  full_series_count  threshold_P95  threshold_P05  anomaly_count_in_window earliest_anomalous_timestamp_in_window  max_value_in_window  min_value_in_window direction
0   frontend-2  mesh.source.frontend.productcatalogservice.ist...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
1   frontend-2  mesh.source.frontend.cartservice.istio_request...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
2   frontend-2  mesh.source.frontend.cartservice.istio_request...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
3   frontend-2  mesh.source.frontend.cartservice.istio_request...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
4   frontend-2  mesh.source.frontend.adservice.istio_request_d...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
5   frontend-2  mesh.source.frontend.adservice.istio_requests....                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
6   frontend-2  mesh.source.frontend.adservice.istio_response_...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
7   frontend-2  mesh.destination.unknown.frontend.istio_respon...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
8   frontend-2  mesh.source.frontend.adservice.istio_request_b...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
9   frontend-2  mesh.destination.unknown.frontend.istio_reques...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
10  frontend-2  mesh.destination.unknown.frontend.istio_reques...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
11  frontend-2  mesh.destination.unknown.frontend.istio_respon...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
12  frontend-2  mesh.destination.unknown.frontend.istio_reques...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
13  frontend-0  mesh.destination.unknown.frontend.istio_reques...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
14  frontend-0         container.node-5.container_spec_cpu_period                 30       100000.0       100000.0                       30              2022-03-20 11:30:00+08:00             100000.0             100000.0      high
15  frontend-0     container.node-5.container_tasks_state.running                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
16  frontend-0    container.node-5.container_tasks_state.sleeping                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
17  frontend-0     container.node-5.container_tasks_state.stopped                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high
18  frontend-0  container.node-5.container_tasks_state.uninter...                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0       low
19  frontend-0             container.node-5.container_threads_max                 30            0.0            0.0                       30              2022-03-20 11:30:00+08:00                  0.0                  0.0      high```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timezone-aware dt (UTC+8) exists for potential follow-ups
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter metric rows that mention 'node' in either cmdb_id or kpi_name (case-insensitive) and store for follow-ups
metric_node_rows = df_metric[
    df_metric['cmdb_id'].str.contains('node', case=False, na=False) |
    df_metric['kpi_name'].str.contains('node', case=False, na=False)
].copy()

# Identify unique cmdb_id values that contain 'node' (case-insensitive) and compute counts and example KPIs
mask_cmdb_node = df_metric['cmdb_id'].str.contains('node', case=False, na=False)
cmdb_node_counts = (
    df_metric[mask_cmdb_node]
    .groupby('cmdb_id')['kpi_name']
    .agg(row_count='count', example_kpis=lambda s: list(pd.Series(s.unique())[:10]))
    .reset_index()
)

# Convert example_kpis list to a compact semi-colon separated string for display
cmdb_node_counts['example_kpis'] = cmdb_node_counts['example_kpis'].apply(lambda lst: '; '.join(lst))

# Select and sort (by row_count desc), limit to top 20
metric_node_summary = cmdb_node_counts[['cmdb_id','row_count','example_kpis']].sort_values('row_count', ascending=False).head(20).reset_index(drop=True)

# Return the compact table (and keep metric_node_rows in kernel for follow-ups)
metric_node_summary

```
Out[7]:
```
Summary of node-level metrics found in metric.csv:

- Nodes detected (sorted by row count):
  - node-2: 1,722 rows
  - node-4: 1,722 rows
  - node-1: 1,692 rows
  - node-3: 1,692 rows
  - node-5: 1,692 rows
  - node-6: 1,692 rows

- Example KPI names (up to 10 per node, shown compactly): ping.can_connect; system.cpu.iowait; system.cpu.pct_usage; system.mem.*; system.disk.* (the stored example list includes similar system- and ping-related KPIs).

- Note: all metric rows containing "node" in cmdb_id or kpi_name are stored in the kernel variable metric_node_rows for follow-up inspection.

Recommendation: prioritize inspecting node-2 and node-4 (highest row counts) for resource or connectivity signals (CPU iowait, CPU usage, ping failures, disk/memory metrics) and correlate with service/pod anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  row_count                                       example_kpis
0  node-2       1722  ping.can_connect; system.cpu.iowait; system.cp...
1  node-4       1722  ping.can_connect; system.cpu.iowait; system.cp...
2  node-1       1692  system.cpu.iowait; system.cpu.pct_usage; syste...
3  node-3       1692  system.cpu.iowait; system.cpu.pct_usage; syste...
4  node-5       1692  system.cpu.iowait; system.cpu.pct_usage; syste...
5  node-6       1692  system.cpu.iowait; system.cpu.pct_usage; syste...```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing df_metric and tz if available; otherwise load and prepare
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware datetime column 'dt' in UTC+8
df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Node targets and incident window (UTC+8)
node_targets = ['node-1','node-2','node-3','node-4','node-5','node-6']
window_start = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
window_end   = pd.Timestamp('2022-03-20 11:59:59', tz=tz)

# Lower-is-bad substrings (case-insensitive)
lower_is_bad_substrs = ['sr','success','ss','rate','rr']
def is_lower_is_bad(name):
    n = str(name).lower()
    return any(sub in n for sub in lower_is_bad_substrs)

# 1) Compute global thresholds (P95 and P05) for node-level (full series)
node_thresholds = (
    df_metric[df_metric['cmdb_id'].isin(node_targets)]
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(full_series_count='count',
         threshold_P95=lambda x: x.quantile(0.95),
         threshold_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric points to the incident window for node targets and merge thresholds
node_metric_window = df_metric[
    (df_metric['cmdb_id'].isin(node_targets)) &
    (df_metric['dt'] >= window_start) &
    (df_metric['dt'] <= window_end)
].copy()

node_metric_window = node_metric_window.merge(node_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Determine direction per kpi_name
node_metric_window['direction'] = node_metric_window['kpi_name'].apply(lambda n: 'low' if is_lower_is_bad(n) else 'high')

# Flag anomalies according to rules: high -> value >= P95 ; low -> value <= P05
def flag_anom(row):
    p95 = row['threshold_P95']
    p05 = row['threshold_P05']
    v = row['value']
    # handle missing thresholds: treat as not anomalous
    if pd.isna(p95) or pd.isna(p05):
        return False
    if row['direction'] == 'high':
        return v >= p95
    else:
        return v <= p05

node_metric_window['anomalous'] = node_metric_window.apply(flag_anom, axis=1)

# Store detailed flagged anomalies DataFrame in kernel for follow-ups
node_metric_anoms_df = node_metric_window.copy()

# 3) Aggregate anomalies per (cmdb_id, kpi_name) and produce compact ranked table
node_anom_summary = (
    node_metric_anoms_df[node_metric_anoms_df['anomalous']]
    .groupby(['cmdb_id','kpi_name','direction'])
    .agg(
        anomaly_count_in_window=('anomalous','sum'),
        earliest_anomalous_timestamp_in_window=('dt','min'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min')
    ).reset_index()
    .merge(node_thresholds, left_on=['cmdb_id','kpi_name'], right_on=['cmdb_id','kpi_name'], how='left')
    .loc[:, [
        'cmdb_id',
        'kpi_name',
        'full_series_count',
        'threshold_P95',
        'threshold_P05',
        'anomaly_count_in_window',
        'earliest_anomalous_timestamp_in_window',
        'max_value_in_window',
        'min_value_in_window',
        'direction'
    ]]
)

# Sort by anomaly_count_in_window desc and keep top 20 (compact)
node_anom_summary_top20 = node_anom_summary.sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

# Display the compact table variable (per instruction, show variable rather than print)
node_anom_summary_top20

```
Out[8]:
```
Summary of node-level anomaly scan for the incident window (2022-03-20 11:30:00 → 11:59:59 UTC+8):

What I did
- Computed global P95 and P05 per (node, kpi) using the full metric series, then flagged points inside the 30‑minute window using the rules you specified (>=P95 for "high", <=P05 for "low").

Top findings (compact)
- Many node-level (cmdb_id, kpi_name) groups were flagged; the top examples (all have full_series_count=30 and anomaly_count_in_window=30, earliest anomalous timestamp = 2022-03-20 11:30:00+08:00) include:
  - node-6 · system.tcp.retrans_pct — anomalies flagged (direction=high)
  - node-6 · system.swap.used_pct, system.swap.used, system.swap.total, system.swap.si, system.swap.so, system.swap.free — all flagged (high)
  - node-6 · system.process.zombie.num — flagged (direction=low)
  - node-6 · system.os.nofile.max — flagged (high)
  - node-6 · system.net.udp.rcv_buf_errors and system.net.udp.snd_buf_errors — flagged (low)
  - node-1 · system.disk.total and system.disk.readonly — flagged (high)
  - node-1 · system.fs.inodes.in_use and system.io.r_await — flagged (high)
- In short: multiple swap, disk, TCP/UDP, and filesystem metrics on nodes (especially node-6 and node-1) appear anomalous in the window.

Important caveat
- Many flagged groups have constant or zero-valued series where P95 == P05 == the observed value (e.g., 0.0 or a fixed integer). That causes every point in the window to meet the anomaly rule, so the result may reflect threshold edge-cases (constant series) rather than meaningful spikes/drops.

Recommended next steps
1. Inspect full-series distributions for the flagged KPIs to confirm whether these are genuine deviations or constant/zero-series artifacts.  
2. Prioritize node-6 and node-1 metrics (highest concentration of flagged KPIs) and check node logs and resource usage (swap, disk, network retransmits).  
3. Correlate node findings with frontend errors/traces to see if node-level issues align temporally with frontend failures.

Follow-up data
- The detailed flagged node-level anomalies are stored in kernel variable node_metric_anoms_df for further analysis and correlation.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       kpi_name  full_series_count  threshold_P95  threshold_P05  anomaly_count_in_window earliest_anomalous_timestamp_in_window  max_value_in_window  min_value_in_window direction
0   node-6         system.tcp.retrans_pct                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
1   node-6           system.swap.used_pct                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
2   node-6               system.swap.used                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
3   node-6              system.swap.total                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
4   node-6                 system.swap.so                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
5   node-6                 system.swap.si                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
6   node-6               system.swap.free                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
7   node-1           system.disk.readonly                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
8   node-1              system.disk.total                 30   5.867011e+09   5.867011e+09                       30              2022-03-20 11:30:00+08:00         5.867011e+09         5.867011e+09      high
9   node-6      system.process.zombie.num                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
10  node-1        system.fs.inodes.in_use                 30   2.500000e-01   2.500000e-01                       30              2022-03-20 11:30:00+08:00         2.500000e-01         2.500000e-01      high
11  node-6           system.os.nofile.max                 30   6.553600e+06   6.553600e+06                       30              2022-03-20 11:30:00+08:00         6.553600e+06         6.553600e+06      high
12  node-6  system.net.udp.rcv_buf_errors                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
13  node-6  system.net.udp.snd_buf_errors                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
14  node-1              system.io.r_await                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
15  node-6           system.disk.readonly                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
16  node-6        system.fs.inodes.in_use                 30   8.900000e-01   8.900000e-01                       30              2022-03-20 11:30:00+08:00         8.900000e-01         8.900000e-01      high
17  node-6              system.disk.total                 30   3.617924e+09   3.617924e+09                       30              2022-03-20 11:30:00+08:00         3.617924e+09         3.617924e+09      high
18  node-1                 system.swap.so                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
19  node-1                 system.swap.si                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse existing variables from kernel: df_metric, node_metric_anoms_df
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    node_metric_anoms_df
except NameError:
    raise RuntimeError("node_metric_anoms_df not found in kernel. Run the node-level anomaly scan step first.")

# Ensure timezone-aware datetime column 'dt' in UTC+8
tz = pytz.timezone('Asia/Shanghai')
df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Focus on node-6 and node-1 flagged KPI groups from the previous anomaly scan
flagged_nodes = ['node-6', 'node-1']
flagged_groups = (
    node_metric_anoms_df[
        node_metric_anoms_df['cmdb_id'].isin(flagged_nodes)
    ][['cmdb_id','kpi_name']]
    .drop_duplicates()
)

# Prepare a filter to select full-series rows for these flagged groups
# Merge to avoid accidental partial string matches
flagged_full_series = df_metric.merge(flagged_groups, on=['cmdb_id','kpi_name'], how='inner')

#  Compute full-series distribution stats per (cmdb_id, kpi_name)
agg_funcs = {
    'full_series_count': ('value', 'count'),
    'number_of_unique_values': ('value', lambda x: x.nunique()),
    'min': ('value', 'min'),
    'P25': ('value', lambda x: x.quantile(0.25)),
    'P50': ('value', lambda x: x.quantile(0.50)),
    'mean': ('value', 'mean'),
    'P75': ('value', lambda x: x.quantile(0.75)),
    'P95': ('value', lambda x: x.quantile(0.95)),
    'max': ('value', 'max'),
    'std': ('value', 'std')
}

dist_table = (
    flagged_full_series
    .groupby(['cmdb_id','kpi_name'])
    .agg(**agg_funcs)
    .reset_index()
)

# Sort by std descending and keep up to 20 rows (compact)
dist_table_sorted = dist_table.sort_values('std', ascending=False).head(20).reset_index(drop=True)

# Store detailed per-point series for top 3 most variable KPIs (by std) for follow-up
top3 = dist_table_sorted.nlargest(3, 'std')[['cmdb_id','kpi_name']]
# node_top3_series: full-series rows for those top 3 groups
node_top3_series = df_metric.merge(top3, on=['cmdb_id','kpi_name'], how='inner').sort_values(['cmdb_id','kpi_name','dt']).reset_index(drop=True)

# Final compact output: distribution table (top 20 by std)
dist_table_sorted

```
Out[9]:
```
Summary of full-series distributions for node-6 and node-1 KPIs that were flagged:

- Top variability (sorted by std, up to 20 rows shown). Key columns: full_series_count, unique values, median, min/max, std.
  - node-1 · system.fs.inodes.free — count=30, unique=30, median≈5.0558e+09, min≈5.0437e+09, max≈5.0656e+09, std≈5.80e+06
  - node-1 · system.fs.inodes.total — count=30, unique=30, median≈5.0577e+09, min≈5.0456e+09, max≈5.0676e+09, std≈5.80e+06
  - node-6 · system.fs.inodes.free — count=30, unique=30, median≈2.9054e+09, min≈2.8983e+09, max≈2.9108e+09, std≈3.28e+06
  - node-6 · system.fs.inodes.total — count=30, unique=30, median≈2.9068e+09, min≈2.8997e+09, max≈2.9122e+09, std≈3.28e+06
  - node-1 · system.disk.used — count=30, unique=29, median≈4.3079e+09, std≈1.45e+06
  - node-1 · system.disk.free — count=30, unique=30, median≈1.5464e+09, std≈1.45e+06
  - node-6 · system.disk.used — count=30, unique=30, median≈2.5314e+09, std≈9.81e+05
  - node-6 · system.disk.free — count=30, unique=30, median≈1.0684e+09, std≈9.81e+05
  - node-1 · system.net.bytes_sent / bytes_rcvd and tcp segments show higher variability in bytes and segments (std hundreds of thousands to millions).
  - Several other metrics (fs inodes used, mem, net counters) have smaller but non-negligible stds.

- Notes and observations:
  - Most groups have 30 full-series samples and 30 unique values, indicating steady per-minute variation across the series (not constant).
  - The largest variability is in filesystem inode counts (node-1 and node-6) and disk used/free values — these show the biggest standard deviations and multi-million ranges.
  - A few metrics have only 6 samples (e.g., system.mem.real.used, system.mem.usable) — treat their stats cautiously.

- Follow-up:
  - I stored the full per-point series for the top 3 most variable KPIs (by std) in the kernel variable node_top3_series for further inspection. The top 3 are:
    1) node-1 · system.fs.inodes.free
    2) node-1 · system.fs.inodes.total
    3) node-6 · system.fs.inodes.free

- Recommendation:
  - Investigate inode and disk usage patterns on node-1 and node-6 (these show the largest variability) and correlate with load or processes creating/deleting many files. Also correlate network bytes/segments variability with traffic during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                 kpi_name  full_series_count  number_of_unique_values           min           P25           P50          mean           P75           P95           max           std
0   node-1    system.fs.inodes.free                 30                       30  5.043688e+09  5.050382e+09  5.055784e+09  5.055129e+09  5.059109e+09  5.063382e+09  5.065621e+09  5.801357e+06
1   node-1   system.fs.inodes.total                 30                       30  5.045624e+09  5.052318e+09  5.057719e+09  5.057064e+09  5.061044e+09  5.065316e+09  5.067555e+09  5.800852e+06
2   node-6    system.fs.inodes.free                 30                       30  2.898298e+09  2.902352e+09  2.905435e+09  2.904960e+09  2.907256e+09  2.909650e+09  2.910758e+09  3.283445e+06
3   node-6   system.fs.inodes.total                 30                       30  2.899696e+09  2.903749e+09  2.906832e+09  2.906357e+09  2.908653e+09  2.911046e+09  2.912155e+09  3.283155e+06
4   node-1         system.disk.used                 30                       29  4.305438e+09  4.307071e+09  4.307893e+09  4.308057e+09  4.309240e+09  4.310228e+09  4.310923e+09  1.449211e+06
5   node-1         system.disk.free                 30                       30  1.543407e+09  1.545090e+09  1.546437e+09  1.546273e+09  1.547258e+09  1.548331e+09  1.548892e+09  1.449193e+06
6   node-6         system.disk.used                 30                       30  2.529858e+09  2.530873e+09  2.531449e+09  2.531583e+09  2.532380e+09  2.533029e+09  2.533500e+09  9.814092e+05
7   node-6         system.disk.free                 30                       30  1.066308e+09  1.067428e+09  1.068360e+09  1.068226e+09  1.068936e+09  1.069647e+09  1.069951e+09  9.813999e+05
8   node-1    system.net.bytes_sent                 30                       30  7.945985e+05  1.040498e+06  1.084816e+06  1.165889e+06  1.126704e+06  1.985069e+06  2.397562e+06  3.816317e+05
9   node-1    system.net.bytes_rcvd                 30                       30  6.993838e+05  9.540464e+05  1.001498e+06  1.013508e+06  1.045887e+06  1.460436e+06  1.713362e+06  2.344043e+05
10  node-1  system.net.tcp.out_segs                 30                       30  3.683350e+03  4.814310e+03  5.147600e+03  5.442642e+03  5.314350e+03  9.187319e+03  1.114653e+04  1.770390e+03
11  node-6    system.net.bytes_rcvd                 30                       30  6.036644e+04  6.407657e+04  6.481414e+04  6.481885e+04  6.569274e+04  6.691139e+04  6.862360e+04  1.620830e+03
12  node-6    system.net.bytes_sent                 30                       30  7.293670e+04  7.632628e+04  7.707176e+04  7.710655e+04  7.811609e+04  7.913179e+04  8.096998e+04  1.611979e+03
13  node-1    system.fs.inodes.used                 30                       30  1.934384e+06  1.934841e+06  1.935178e+06  1.935195e+06  1.935584e+06  1.935969e+06  1.936005e+06  4.965995e+02
14  node-1          system.mem.free                 30                       30  2.915000e+03  3.227250e+03  3.438000e+03  3.474167e+03  3.793750e+03  3.947250e+03  4.035000e+03  3.239270e+02
15  node-6    system.fs.inodes.used                 30                       30  1.396538e+06  1.396824e+06  1.397004e+06  1.397023e+06  1.397246e+06  1.397459e+06  1.397508e+06  2.846641e+02
16  node-1     system.mem.real.used                  6                        6  2.363400e+04  2.372800e+04  2.389250e+04  2.393383e+04  2.411400e+04  2.428425e+04  2.432300e+04  2.700677e+02
17  node-1   system.net.tcp.in_segs                 30                       30  1.129170e+03  1.283058e+03  1.470710e+03  1.472273e+03  1.522587e+03  1.982343e+03  2.306220e+03  2.699156e+02
18  node-1        system.mem.usable                  6                        6  7.113000e+03  7.322000e+03  7.543500e+03  7.502000e+03  7.707250e+03  7.787000e+03  7.802000e+03  2.698896e+02
19  node-1          system.mem.used                 30                       30  2.414400e+04  2.435975e+04  2.453550e+04  2.454927e+04  2.473575e+04  2.492855e+04  2.497300e+04  2.412613e+02```
```

In [ ]:
# In[10]:

import pandas as pd
import re

# Reuse df_metric from kernel
targets = ["frontend-0","frontend-1","frontend-2"]

# Filter metric rows for the three frontends and kpi_names containing 'container.node-' (case-sensitive)
metric_frontend_node_rows = df_metric[
    df_metric['cmdb_id'].isin(targets) &
    df_metric['kpi_name'].str.contains('container.node-', na=False)
].copy()

# Extract node identifier (e.g., 'node-5') from kpi_name using regex
metric_frontend_node_rows['node_num'] = metric_frontend_node_rows['kpi_name'].str.extract(r'container\.node-(\d+)', expand=False)
metric_frontend_node_rows['node_id'] = metric_frontend_node_rows['node_num'].apply(lambda x: f'node-{x}' if pd.notna(x) else None)

# Drop rows where extraction failed (safety)
metric_frontend_node_rows = metric_frontend_node_rows[metric_frontend_node_rows['node_id'].notna()]

# Count distinct kpi_name per (cmdb_id, node_id)
counts = (
    metric_frontend_node_rows
    .groupby(['cmdb_id','node_id'])['kpi_name']
    .nunique()
    .reset_index(name='metric_count')
)

# Build compact summary per frontend: list of unique node_ids (up to 10) and small mapping node->count
def build_summary(df_grp):
    # sort nodes by metric_count desc for clarity
    rows = df_grp.sort_values('metric_count', ascending=False)
    node_list = rows['node_id'].unique()[:10]
    node_list_str = '; '.join(node_list)
    mapping_str = '; '.join([f"{r['node_id']}: {int(r['metric_count'])}" for _, r in rows.iterrows()])
    return pd.Series({'list_of_unique_node_ids_referenced': node_list_str, 'count_of_metrics_referencing_each_node': mapping_str})

summary = counts.groupby('cmdb_id').apply(build_summary).reset_index()

# Ensure only the three frontends are present and return compact table
frontend_node_reference_summary = summary[summary['cmdb_id'].isin(targets)].reset_index(drop=True)

# Display the compact result (and keep metric_frontend_node_rows in kernel for follow-ups)
frontend_node_reference_summary

```
Out[10]:
```
Summary of container.node- references for the three frontends:

- frontend-0: references node-5 (unique nodes: node-5). Metrics referencing node-5: 64.
- frontend-1: references node-5 (unique nodes: node-5). Metrics referencing node-5: 64.
- frontend-2: references node-5 (unique nodes: node-5). Metrics referencing node-5: 64.

Interpretation: all three frontend components have container KPIs that reference the same node (node-5) — 64 distinct container.node-* metrics per frontend. Suggest investigating node-5 (resource and network metrics, logs) as it is implicated across all frontends.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id list_of_unique_node_ids_referenced count_of_metrics_referencing_each_node
0  frontend-0                             node-5                             node-5: 64
1  frontend-1                             node-5                             node-5: 64
2  frontend-2                             node-5                             node-5: 64```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

# Reuse df_metric and tz if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware datetime column 'dt' in UTC+8
df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target node
node_target = 'node-5'

# Incident window (inclusive) in UTC+8
window_start = pd.Timestamp('2022-03-20 11:30:00', tz=tz)
window_end   = pd.Timestamp('2022-03-20 11:59:59', tz=tz)

# Lower-is-bad substrings (case-insensitive)
lower_is_bad_substrs = ['sr','success','ss','rate','rr']
def is_lower_is_bad(name):
    n = str(name).lower()
    return any(sub in n for sub in lower_is_bad_substrs)

# 1) Compute global thresholds (P95 and P05) for node-5 using full series (no time filter)
node5_thresholds = (
    df_metric[df_metric['cmdb_id'] == node_target]
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(full_series_count='count',
         threshold_P95=lambda x: x.quantile(0.95),
         threshold_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter node-5 points to the incident window and merge thresholds
node5_window = df_metric[
    (df_metric['cmdb_id'] == node_target) &
    (df_metric['dt'] >= window_start) &
    (df_metric['dt'] <= window_end)
].copy()

node5_window = node5_window.merge(node5_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Determine direction per kpi_name
node5_window['direction'] = node5_window['kpi_name'].apply(lambda n: 'low' if is_lower_is_bad(n) else 'high')

# Flag anomalies: high -> value >= P95 ; low -> value <= P05
def flag_row(r):
    p95 = r['threshold_P95']
    p05 = r['threshold_P05']
    v = r['value']
    if pd.isna(p95) or pd.isna(p05):
        return False
    return (v >= p95) if r['direction']=='high' else (v <= p05)

node5_window['anomalous'] = node5_window.apply(flag_row, axis=1)

# 4) Store detailed flagged node-5 anomalies for follow-ups
node5_metric_anoms_df = node5_window.copy()

# Aggregate anomalies per (cmdb_id, kpi_name)
node5_anom_summary = (
    node5_metric_anoms_df[node5_metric_anoms_df['anomalous']]
    .groupby(['cmdb_id','kpi_name','direction'])
    .agg(
        anomaly_count_in_window=('anomalous','sum'),
        earliest_anomalous_timestamp_in_window=('dt','min'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min')
    )
    .reset_index()
    .merge(node5_thresholds, on=['cmdb_id','kpi_name'], how='left')
    .loc[:, [
        'cmdb_id',
        'kpi_name',
        'full_series_count',
        'threshold_P95',
        'threshold_P05',
        'anomaly_count_in_window',
        'earliest_anomalous_timestamp_in_window',
        'max_value_in_window',
        'min_value_in_window',
        'direction'
    ]]
)

# Compact ranked table: sort by anomaly_count_in_window desc and keep top 20
node5_anom_summary_top20 = node5_anom_summary.sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

# Return the compact table variable only
node5_anom_summary_top20

```
Out[11]:
```
Summary of node-5 anomaly scan for the incident window (2022-03-20 11:30:00 → 11:59:59 UTC+8):

- What was done: global P95/P05 thresholds were computed using the entire series per (node-5, kpi). Points inside the 30-minute window were flagged as anomalous using the given high/low rules.

- Key results (compact):
  - Many node-5 metrics were flagged; the top examples (up to 20 shown) all have:
    - full_series_count = 30 (the full file contains 30 samples per metric),
    - anomaly_count_in_window = 30 (every point in the window was flagged),
    - earliest anomalous timestamp = 2022-03-20 11:30:00+08:00.
  - Flagged KPI types include disk and filesystem metrics (system.disk.readonly, system.disk.total, system.fs.inodes.total, system.fs.inodes.in_use), I/O metrics (system.io.r_await, system.io.r_s, system.io.rkb_s), swap metrics (system.swap.*), OS limits (system.os.nofile.max), network error counters (system.net.packets_in.error, system.net.packets_out.error, system.net.udp.*), memory total, process.zombie.num, etc.
  - Directions: most are labeled "high" by the rule; a few network/error counters are treated "low" (per substring rules) or flagged low because thresholds equal observed values.

- Important caveat:
  - Many thresholds (P95 or P05) are identical to the observed constant values (often 0.0 or a fixed integer), so every point in the window meets the anomaly condition. This indicates threshold edge-cases (constant/zero-valued series) rather than clear spikes or drops.

- Recommended next steps:
  1. Inspect the full series and variance for the flagged KPIs in node5_metric_anoms_df (stored in the kernel) to determine whether flags reflect real anomalies or constant-value artifacts.  
  2. Prioritize investigation of node-5 disk/fs usage, swap, and network error counters (they appear repeatedly flagged). Check node logs, disk health, inode usage, and network interface errors.  
  3. Correlate node-5 findings with frontend errors (frontends reference node-5) and traces to determine impact and root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       kpi_name  full_series_count  threshold_P95  threshold_P05  anomaly_count_in_window earliest_anomalous_timestamp_in_window  max_value_in_window  min_value_in_window direction
0   node-5           system.disk.readonly                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
1   node-5              system.disk.total                 30   6.191419e+08   6.191419e+08                       30              2022-03-20 11:30:00+08:00         6.191419e+08         6.191419e+08      high
2   node-5              system.io.r_await                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
3   node-5                  system.io.r_s                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
4   node-5         system.fs.inodes.total                 30   3.932160e+07   3.932160e+07                       30              2022-03-20 11:30:00+08:00         3.932160e+07         3.932160e+07      high
5   node-5                system.io.rkb_s                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
6   node-5        system.fs.inodes.in_use                 30   8.200000e-01   8.200000e-01                       30              2022-03-20 11:30:00+08:00         8.200000e-01         8.200000e-01      high
7   node-5   system.net.packets_out.error                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
8   node-5    system.net.packets_in.error                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
9   node-5               system.mem.total                 30   3.201100e+04   3.201100e+04                       30              2022-03-20 11:30:00+08:00         3.201100e+04         3.201100e+04      high
10  node-5               system.swap.used                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
11  node-5  system.net.udp.rcv_buf_errors                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
12  node-5  system.net.udp.snd_buf_errors                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
13  node-5           system.os.nofile.max                 30   6.553600e+06   6.553600e+06                       30              2022-03-20 11:30:00+08:00         6.553600e+06         6.553600e+06      high
14  node-5       system.net.udp.in_errors                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
15  node-5                 system.swap.so                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
16  node-5                 system.swap.si                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
17  node-5               system.swap.free                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high
18  node-5      system.process.zombie.num                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00       low
19  node-5           system.swap.used_pct                 30   0.000000e+00   0.000000e+00                       30              2022-03-20 11:30:00+08:00         0.000000e+00         0.000000e+00      high```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

# Reuse existing kernel variables
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    node5_metric_anoms_df
except NameError:
    raise RuntimeError("node5_metric_anoms_df not found in kernel. Please run the node-5 anomaly scan step first.")

# Ensure timezone-aware dt column (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Determine the set of node-5 KPI names that were flagged (unique kpi_name where anomalous==True)
flagged_kpis_node5 = node5_metric_anoms_df[node5_metric_anoms_df['anomalous'] == True]['kpi_name'].unique().tolist()

# If none flagged, produce empty DataFrame structure
if len(flagged_kpis_node5) == 0:
    dist_table_sorted = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','full_series_count','number_of_unique_values','min','P25','P50','mean','P75','P95','max','std'
    ])
else:
    # 2) Select full-series rows from df_metric for node-5 and the flagged KPI names (no time filtering)
    node5_full_series_flagged = df_metric[(df_metric['cmdb_id'] == 'node-5') & (df_metric['kpi_name'].isin(flagged_kpis_node5))].copy()

    # 3) Compute distribution statistics per (cmdb_id, kpi_name) using the entire series
    agg_funcs = {
        'full_series_count': ('value', 'count'),
        'number_of_unique_values': ('value', lambda x: x.nunique()),
        'min': ('value', 'min'),
        'P25': ('value', lambda x: x.quantile(0.25)),
        'P50': ('value', lambda x: x.quantile(0.50)),
        'mean': ('value', 'mean'),
        'P75': ('value', lambda x: x.quantile(0.75)),
        'P95': ('value', lambda x: x.quantile(0.95)),
        'max': ('value', 'max'),
        'std': ('value', 'std')
    }

    dist_table = (
        node5_full_series_flagged
        .groupby(['cmdb_id','kpi_name'])
        .agg(**agg_funcs)
        .reset_index()
    )

    # 4) Sort by std descending and keep up to 20 rows
    dist_table_sorted = dist_table.sort_values('std', ascending=False).head(20).reset_index(drop=True)

    # 5) Store full per-point series for top 3 most variable KPIs by std in kernel variable node5_top3_series
    top3_kpis = dist_table_sorted.nlargest(3, 'std')[['kpi_name']].kpi_name.tolist()
    node5_top3_series = df_metric[(df_metric['cmdb_id'] == 'node-5') & (df_metric['kpi_name'].isin(top3_kpis))].sort_values(['kpi_name','dt']).reset_index(drop=True)

# Final compact output: distribution table (up to 20 rows)
dist_table_sorted

```
Out[12]:
```
Summary of full-series distributions for node-5 KPIs that were flagged:

- Top variability (sorted by standard deviation, up to 20 rows shown). All groups use the entire metric file (no time filtering):
  - system.disk.free — count=30, unique=30, median≈549,958,300, std≈48,162
  - system.disk.used — count=30, unique=30, median≈43,821,520, std≈48,158
  - system.net.bytes_rcvd — count=30, unique=30, median≈25,968, std≈635
  - system.net.bytes_sent — count=30, unique=30, median≈40,994, std≈529
  - system.mem.free — count=30, unique=28, median≈16,563, std≈54
  - system.os.nofile.current — count=30, unique=5, median≈14,096, std≈40
  - system.io.w_s — count=30, unique=11, median=0, mean≈11.1, std≈22.5
  - system.mem.used — count=30, unique=21, median≈7,992.5, std≈11.35
  - ... (other metrics follow with smaller std; see table for full top-20)

- Notes:
  - All listed KPIs have 30 full-series samples (most per-minute samples across the collection period).
  - The largest variability is in disk free/used values and in some network byte counters — these are the most volatile metrics on node-5.
  - Some metrics show few unique values (e.g., os.nofile.current has only 5 distinct values), others vary every sample.

- Follow-up data:
  - I stored the full per-point series for the top 3 most variable KPIs (by std) in the kernel variable node5_top3_series for deeper inspection. The top 3 are:
    1) system.disk.free
    2) system.disk.used
    3) system.net.bytes_rcvd

- Recommendation:
  - Investigate disk usage patterns (free/used) and network traffic on node-5 during the incident window and across the full series to understand causes of variability and any correlation with frontend failures.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                      kpi_name  full_series_count  number_of_unique_values           min           P25           P50          mean           P75           P95           max           std
0   node-5              system.disk.free                 30                       30  5.498981e+08  5.499279e+08  5.499583e+08  5.499678e+08  5.500051e+08  5.500487e+08  5.500534e+08  48162.805756
1   node-5              system.disk.used                 30                       30  4.372640e+07  4.377476e+07  4.382152e+07  4.381203e+07  4.385199e+07  4.387074e+07  4.388172e+07  48157.631645
2   node-5         system.net.bytes_rcvd                 30                       30  2.507559e+04  2.574634e+04  2.596842e+04  2.616653e+04  2.649458e+04  2.722768e+04  2.764736e+04    634.980041
3   node-5         system.net.bytes_sent                 30                       30  3.979985e+04  4.059240e+04  4.099488e+04  4.094153e+04  4.137995e+04  4.164762e+04  4.182228e+04    529.126226
4   node-5               system.mem.free                 30                       28  1.649200e+04  1.652525e+04  1.656300e+04  1.657043e+04  1.661550e+04  1.665520e+04  1.665900e+04     54.487381
5   node-5      system.os.nofile.current                 30                        5  1.404800e+04  1.404800e+04  1.409600e+04  1.409387e+04  1.411200e+04  1.414400e+04  1.417600e+04     40.023901
6   node-5                 system.io.w_s                 30                       11  0.000000e+00  0.000000e+00  0.000000e+00  1.110000e+01  1.612500e+01  6.702500e+01  8.500000e+01     22.500728
7   node-5               system.mem.used                 30                       21  7.981000e+03  7.989000e+03  7.992500e+03  7.997700e+03  8.004500e+03  8.017550e+03  8.026000e+03     11.353717
8   node-5          system.mem.real.used                  6                        6  6.835000e+03  6.837750e+03  6.843500e+03  6.846000e+03  6.851500e+03  6.861250e+03  6.864000e+03     11.063453
9   node-5             system.mem.usable                  6                        6  2.399700e+04  2.400725e+04  2.401550e+04  2.401333e+04  2.402150e+04  2.402350e+04  2.402400e+04     10.576704
10  node-5       system.net.tcp.out_segs                 30                       30  2.021700e+02  2.053225e+02  2.085250e+02  2.092557e+02  2.123725e+02  2.174710e+02  2.211300e+02      5.089687
11  node-5        system.udp.connect.num                 30                       12  1.000000e+01  1.800000e+01  2.200000e+01  1.986667e+01  2.300000e+01  2.400000e+01  2.400000e+01      4.133407
12  node-5  system.net.udp.out_datagrams                 30                       30  1.020300e+02  1.077825e+02  1.103150e+02  1.103553e+02  1.127250e+02  1.162525e+02  1.177700e+02      3.828851
13  node-5   system.net.udp.in_datagrams                 30                       30  1.020300e+02  1.077825e+02  1.103150e+02  1.103553e+02  1.127250e+02  1.162525e+02  1.177700e+02      3.828851
14  node-5        system.net.tcp.in_segs                 30                       28  1.645100e+02  1.671250e+02  1.689550e+02  1.692667e+02  1.699000e+02  1.750255e+02  1.826200e+02      3.661723
15  node-5         system.fs.inodes.free                 30                        2  3.899856e+07  3.899856e+07  3.899856e+07  3.899856e+07  3.899856e+07  3.899856e+07  3.899856e+07      1.864366
16  node-5             system.io.w_await                 30                       11  0.000000e+00  0.000000e+00  0.000000e+00  8.570000e-01  1.710000e+00  3.334500e+00  6.500000e+00      1.508571
17  node-5               system.io.await                 30                       11  0.000000e+00  0.000000e+00  0.000000e+00  8.570000e-01  1.710000e+00  3.334500e+00  6.500000e+00      1.508571
18  node-5         system.fs.inodes.used                 30                        3  3.230360e+05  3.230360e+05  3.230360e+05  3.230367e+05  3.230380e+05  3.230380e+05  3.230400e+05      1.093345
19  node-5  system.net.packets_out.count                 30                       28  4.461000e+01  4.598000e+01  4.656500e+01  4.658933e+01  4.742750e+01  4.779550e+01  4.817000e+01      0.918980```
```